### Validamos base

In [ ]:

import sys 
from sqlalchemy import create_engine, text

import numpy as np
import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from funciones import *
from variables_inicio import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "odin"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_odin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)



fecha_mes_base='2026-09-01'

In [2]:

query = f"""
	SELECT *
    FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-09-01'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)

In [3]:

ruta_archivo = os.path.join(ruta_alfin, 'bloqueado_Hoy_y_por_bloquear.csv')
df_correo_ref.to_csv(ruta_archivo, sep=';')


In [4]:

query = f"""
	SELECT dni_cliente as Dni,'1' as ref
    FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-09-01'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)

ruta_archivo = os.path.join(ruta_alfin, 'envio_subir.csv')
df_correo_ref.to_csv(ruta_archivo, sep=',')

filename='envio_subir.csv'

df_correo_01=cargar_archivo_csv_ruta(spark,filename,',',True,ruta_alfin)

In [5]:

filename='para_bloquear_2.csv'
df_boqueo=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='Consulta_de_Campañas_202608_V3_SS_02.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_01.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_03.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_04.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_03=df_validar_03.drop('TASA_MIN_DESCUENTO')
df_validar_04=df_validar_04.drop('TASA_MIN_DESCUENTO')

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)

df_validar=df_validar.dropDuplicates(['DNI'])
df_boqueo=df_boqueo.withColumnRenamed('dni_cliente','DNI')

In [6]:
df_boqueo_consult_camp=df_boqueo.join(df_validar,['DNI'],'inner')
df_boqueo_consult_camp=df_boqueo_consult_camp.dropDuplicates(['DNI'])
df_boqueo_consult_camp.count()

31351

In [ ]:
['DNI', 'nombres', 'celular', 'agencia', 'monto', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO']


['DNI', 'nombres', 'celular', 'agencia', 'monto', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO']


In [41]:
df_boqueo_consult_camp1=df_boqueo_consult_camp1.drop('_c5')

In [9]:
df_boqueo_consult_camp1=df_boqueo_consult_camp.filter(F.col('OFERTA_MAX')>=5000)
df_boqueo_consult_camp1.count()


27295

In [15]:
df_boqueo_pd=df_boqueo_consult_camp1.toPandas()

In [16]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)




dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())



C:\Users\Data\AppData\Local\Temp\ipykernel_14476\1100902651.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [17]:

query = f"""
	SELECT dni_cliente
    FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-09-01'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)
correo_ref = set(df_correo_ref['dni_cliente'].dropna())


In [18]:


df_boqueo_pd=df_boqueo_pd[
    ~df_boqueo_pd['DNI'].isin(correo_ref)&
    ~df_boqueo_pd['DNI'].isin(dni_retiro)&
    ~df_boqueo_pd['celular'].isin(cel_retiro)&
    ~df_boqueo_pd['DNI'].isin(dni_desembolso)
    ].copy()
df_boqueo_pd.shape

(11894, 45)

In [11]:
query = """
    select *,NUMERO_DOCUMENTO as Dni   
    from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where cl_telf1 is not null
    and fecha_envio>='2026-09-01'
    AND RETIRO = 'ACTIVO'
    and cruce='CET'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

query = """
	SELECT distinct Dni FROM THOTH.dbo.Tmp_LLamadas_Alfin_5 
    where Descripcion_ in(
        'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
        'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES',
        'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
        'FUERA DE SERVICIO'
    )
    """
df_quitar=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = """
    SELECT Dni FROM THOTH.dbo.Tmp_LLamadas_Alfin_5 
    where Numero_Campana='401'
    and list_description<>'provicional'
    """
df_quitar2=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

print(df_formato.count())
print(df_quitar.count())
print(df_quitar2.count())

10555
115
0


In [12]:
df_formato=df_formato.join(df_quitar, ['Dni'], "left_anti")
df_formato=df_formato.join(df_quitar2, ['Dni'], "left_anti")

In [13]:
print(df_formato.count())

df_formato=df_formato.join(df_correo_01, ['Dni'], "left_anti")
print(df_formato.count())




10538
3038


In [ ]:
['Dni', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'SERVICIO', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FLAT2', 'REP1', 'REP2', 'PILOTO_RETENCION', 'CAMP_BONO', 'ACCION']colo

['Dni', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RAN

In [19]:

df_boqueo_pd["OFERTA_MAX"] = pd.to_numeric(
    df_boqueo_pd["OFERTA_MAX"],
    errors="coerce"
)
df_boqueo_pd = df_boqueo_pd[
    df_boqueo_pd["OFERTA_MAX"] >= 5000
].copy()

df_boqueo_pd.shape

(11894, 45)

In [24]:
df_boqueo_p1d=df_boqueo_pd[df_boqueo_pd['USER_V3'].isin(['7. Peers','4. MES + PLD No Peers','2. sunedu & sunarp B','2. sunedu & sunarp B','1. sunedu & sunarp A','1. sunedu & sunarp A','8. Tarjetero Cash'])].copy()

In [20]:
df_boqueo_pd['USER_V3'].unique()

array(['11. Dependiente Banca', '7. Peers', '1. sunedu & sunarp A',
       '3. MES + PLD Peers', '8. Tarjetero Cash', '5. MES A',
       '4. MES + PLD No Peers', '9. Dependiente BN',
       '2. sunedu & sunarp B', '6. MES B', '14. Otros Bancarizados',
       '12. Independiente A', '10. Dependiente + Convenios',
       '13. No dependiente + Convenios'], dtype=object)

In [20]:
df_formato=df_formato.filter(F.col('color_final').isin('VERDE OSCURO','VERDE CLARO','','AMARILLO OSCURO'))

In [ ]:
df_formato=df_formato.filter(~F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','2. sunedu & sunarp B','1. sunedu & sunarp A'))


### validar lo que se va a subir 

In [21]:

ruta_archivo = os.path.join(ruta_alfin, 'subir.csv')
df_boqueo_pd.to_csv(ruta_archivo, sep=';')


In [22]:
# Valores a filtrar
colores = ["VERDE CLARO", "VERDE OSCURO"]
campanas = ["RECONQUISTA","REE REH","SOLODNI"]
propension = ['1', '2', '3', '4', '5']
frescura = ['0','1', '2','4']
# user_v3 = ["7. Peers",'3. MES + PLD Peers']
user_v3 = ["7. Peers"]
# Aplicar filtros
df_filtrado = df_boqueo_pd[
    # df_boqueo_pd["COLOR_FINAL"].isin(colores)
    # & df_boqueo_pd["campaña"].isin(campanas)
     df_boqueo_pd["PROPENSION_DISTRIBUCION"].isin(propension)
    # & df_boqueo_pd["FRESCURA"].isin(frescura)
    & df_boqueo_pd["USER_V3"].isin(user_v3)
].copy()

df_filtrado.shape

(5142, 45)

In [121]:
df_boqueo_pd['USER_V3'].unique()

array(['6. MES B', '14. Otros Bancarizados', '1. sunedu & sunarp A',
       '11. Dependiente Banca', '3. MES + PLD Peers', '8. Tarjetero Cash',
       '2. sunedu & sunarp B', '4. MES + PLD No Peers',
       '9. Dependiente BN', '10. Dependiente + Convenios',
       '12. Independiente A', '7. Peers',
       '13. No dependiente + Convenios', '5. MES A'], dtype=object)

In [23]:

ruta_archivo = os.path.join(ruta_alfin, 'tines_que_subir_esto.csv')
df_filtrado.to_csv(ruta_archivo, sep=';')


In [38]:
df_boqueo_p1d.shape

(3314, 44)

In [24]:
df_f=cargar_archivo_csv_ruta(spark,'tines_que_subir_esto.csv',';',True,ruta_alfin)

In [25]:
df_f=df_f.dropDuplicates(['DNI'])
df_f.count()

5142

In [21]:
print(df_f.columns)

['_c0', 'DNI', 'nombres', 'celular', 'agencia', 'monto', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO']


In [ ]:
['_c0', 'DNI', 'nombres', 'celular', 'agencia', 'monto', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']


In [26]:
df_formato_pd=df_f.select(F.col('DNI').alias('dni_cliente'),
F.col('COLOR_FINAL').alias('color'),
F.col('agencia').alias('agencia_atencion'),
F.col('celular').alias('celular'),
F.col('celular').alias('telefono_cliente'),
F.col('OFERTA_MAX').alias('monto_solicitado'),
F.col('nombres').alias('nombre_cliente')).toPandas()

In [27]:
df_formato_pd['dni_vendedor']='00000001'
df_formato_pd['cdv_alfin_banco']='ROSA HONOR'
df_formato_pd['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato_pd['ejecutivo_target']='BOT'
df_formato_pd['codigo_ejecutivo_id']='00000001'
df_formato_pd['operador']='TARGET'
df_formato_pd['tipo_gestion']='Derivacion'
df_formato_pd['tipo_carga']='MANUAL'
df_formato_pd['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'


In [28]:
fechas = pd.to_datetime([
    "2026-09-05",
    "2026-09-06",
    "2026-09-08",
    "2026-09-09",
    "2026-09-10",
])

df_formato_pd["fecha_visita"] = np.random.choice(
    fechas,
    size=len(df_formato_pd)
)

horas = np.random.randint(9, 19, size=len(df_formato_pd))

minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato_pd))

df_formato_pd["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]



In [29]:
df_formato_pd.shape

(5142, 18)

In [30]:
query = f"""
	select *,agencia_correo as agencia_atencion ,agencia_Formulario as agencia_tienda from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)
df_agencia.drop_duplicates(subset=["agencia_atencion"], inplace=True)

df_agencia["agencia_atencion"] = df_agencia["agencia_atencion"].str.strip()
df_formato_pd["agencia_atencion"] = df_formato_pd["agencia_atencion"].str.strip()



equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'JULIACA': 'JULIACA 2',
    'VILLA EL SALVADOR': 'VILLA EL SALVADOR 2',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato_pd['agencia_atencion'] = (
    df_formato_pd['agencia_atencion']
    .replace(equivalencias)
)
# df_formato_pd.drop_duplicates(subset=["dni"], inplace=True)



set_correo = set(
    df_formato_pd['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_atencion']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )


set()
{'NULL'}


In [27]:
df_agencia[
    df_agencia["agencia_atencion"].str.contains("JULIACA", case=False, na=False)
]

,agencia_Formulario,agencia_base,agencia_base2,agencia_correo,correos,agencia_atencion,agencia_tienda
20,735986 - JULIACA 2,JULIACA 2,JULIACA 2,JULIACA 2,Mary.Choque@alfinbanco.pe,JULIACA 2,735986 - JULIACA 2


In [35]:
df_formato_pd=df_formato_pd[
    df_formato_pd["agencia_atencion"] != "JULIACA 2"
].copy()

In [ ]:
JULIACA 2

In [36]:
df_formato_pd=df_formato_pd.merge(df_agencia, on='agencia_atencion', how='left')
df_formato_pd=df_formato_pd.drop_duplicates(subset=["dni_cliente"])


In [37]:
df_correo=df_formato_pd[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()

df_formulario=df_formato_pd[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,10008235,YENI AGUILAR,21100,982337819,ATE VITARTE,2026-09-09,13:45:00,VERDE OSCURO
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,10009357,BARRIENTOS ANYOSA GISELLE ESTRELLA,15100,996670078,SAN JUAN DE MIRAFLORES,2026-09-05,09:00:00,VERDE CLARO


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,10008235,YENI AGUILAR,982337819,739629 - ATE VITARTE,2026-09-09,21100,Derivacion
1,00000001,TARGET,10009357,BARRIENTOS ANYOSA GISELLE ESTRELLA,996670078,738224 - SAN JUAN DE MIRAFLORES,2026-09-05,15100,Derivacion


In [33]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


In [34]:

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)




dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


C:\Users\Data\AppData\Local\Temp\ipykernel_8956\3001280970.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [35]:

df_formato_pd=df_formato_pd[
    ~df_formato_pd['dni_cliente'].isin(dni_retiro)&
    ~df_formato_pd['celular'].isin(cel_retiro)&
    ~df_formato_pd['dni_cliente'].isin(dni_desembolso)
    ].copy()
df_formato_pd.shape


(4474, 24)

In [38]:
df_formato_pd=df_formato_pd.drop_duplicates(subset=["dni_cliente"])

In [39]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

5106

In [136]:

ruta_archivo = os.path.join(ruta_alfin, 'subir.csv')
df_formato_pd.to_csv(ruta_archivo, sep=';')


In [43]:

engine_mysql_1 = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

In [47]:
query = """
SELECT *
FROM crm_target.alfin_clientes
WHERE cl_base = 'setiembre 2026'
"""

chunks = pd.read_sql(
    query,
    engine_mysql_1,
    chunksize=10000
)

df_valentina = pd.concat(chunks, ignore_index=True)

In [46]:
df_valentina

,COUNT(*)
0,60408


In [ ]:
dni_retiro = set(df_formato_pd['dni_cliente'].dropna())


In [54]:
df_boqueo_pd=df_boqueo_pd[
    df_boqueo_pd['DNI'].isin(dni_retiro)
    ].copy()
df_boqueo_pd.shape

(5106, 45)

In [ ]:
df_formato_pd=df_formato_pd[]

In [55]:
df_boqueo_pd['cl_base']='setiembre 2026'

In [ ]:
columnas = [
    'DNI', 'nombres', 'celular', 'agencia','COLOR_FINAL',
    'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO',
    'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades',
    'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2',
    'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD',
    'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE',
    'PROPENSION_DISTRIBUCION', 'OFERTA_SS',
    'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'cl_base'
]

df_boqueo_pd1 = (
    df_boqueo_pd.loc[:, columnas]
    .rename(columns={
        "DNI": "NUMERO_DOCUMENTO",
        "nombres": "NOMBRES",
        "celular": "cl_telf1",
        "COLOR_FINAL": "color_final",
        "TASA_1": "Tasa_1",
        "TASA_2": "Tasa_2",
        "TASA_3": "Tasa_3",
        "TASA_4": "Tasa_4",
        "TASA_5": "Tasa_5",
        "TASA_6": "Tasa_6",
        "TASA_7": "Tasa_7",
        "campaña": "campania",
        "PROPENSION_DISTRIBUCION": "PROPENSION_IC",
        "agencia": "Agencia_comercial"
    })
)

In [63]:
vale=set(df_valentina.columns.tolist())
bloqueo=set(df_boqueo_pd1.columns.tolist())
print(bloqueo-vale)

{'TASA_CREDITO_ANTERIOR', 'FEN', 'rango_deuda', 'PERFIL_ESPECIAL', 'OFERTA_SS', 'PROPENSION_DISTRIBUCION', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'TIPO', 'MARCA_PD', 'TOTAL_A_LIQUIDAR', 'numentidades', 'ALERTA_MAQUETA', 'COD_USER_V3'}


In [67]:
df_boqueo_pd1['PROPENSION_DISTRIBUCION'].unique()

array(['1', '2', '3', '4', '5'], dtype=object)

In [ ]:
['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'cl_estado', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'tasa_minima', 'MES_DURACION_BASE', 'ANIO_DURACION_BASE', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'BLOQUE', 'INTENSIDAD_MAX']tip

['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', '

In [ ]:
df_valentina['TIPO_CLIENTE'].unique()



In [ ]:

# df_prospectos_correos_alfin=df_prospectos_correos_alfin[
#     ~df_prospectos_correos_alfin['dni_cliente'].isin(dni_retiro)&
#     ~df_prospectos_correos_alfin['celular'].isin(cel_retiro)&
#     ~df_prospectos_correos_alfin['dni_cliente'].isin(set_tipi)&
#     ~df_prospectos_correos_alfin['dni_cliente'].isin(dni_desembolso)
#     ].copy()
# df_prospectos_correos_alfin.shape


# dni_retiro = set(df_retiros['dni_cliente'].dropna())
# cel_retiro = set(df_retiros['celular'].dropna())
# dni_desembolso = set(df_desembolso['dni_cliente'].dropna())